## This script 
processes a set of calibrated FITS files for a given standard star and computes its altitude, zenith angle, and airmass at the time each exposure was taken.

Using the observatory’s geographic coordinates and the star’s cataloged RA/Dec, the
script:
  • Reads DATE-OBS timestamps from each FITS header (in local Iran time),
  • Converts them to UTC,
  • Transforms the star’s celestial coordinates to the AltAz frame at the observation time,
  • Calculates altitude, zenith distance, and airmass,
  • Keeps only exposures where the star is above the horizon,
  • Prints a summary table of all valid frames.

IERS auto-download is disabled to ensure predictable time conversion using existing
local IERS data.

In [1]:
from astropy.utils import iers
from astropy.io import fits
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
from astropy.utils import iers
import astropy.units as u
import os
from glob import glob

# -----------------------------
# IERS configuration
# -----------------------------
iers.conf.auto_download = False
iers.conf.auto_max_age = None  # allow predictive data for UT1-UTC

# -----------------------------
# Observatory location
# -----------------------------
obs_lat = 33 + 40/60 + 28/3600  # degrees
obs_lon = 51 + 19/60 + 8/3600  # degrees
obs_height = 1  # meters, approximate if unknown
obs_loc = EarthLocation(lat=obs_lat*u.deg, lon=obs_lon*u.deg, height=obs_height*u.m)

# -----------------------------
# Star info (RA/Dec)
# -----------------------------
star_coords = {
    'BD+35_3659': SkyCoord('19h31m09.22s', '+36d09m10.1s', frame='icrs'),
}

# -----------------------------
# Main processing
# -----------------------------
base_folder = r'C:/Users/Observatory/Exo/Data/Standard-stars/BD353659/06_01_2025/BD353659/calibrated'

all_results = []

for star_folder in star_coords.keys():
    fits_files = glob(os.path.join(base_folder, '*.fits'))

    print(f"\nProcessing star {star_folder}: {len(fits_files)} files found")

    star_results = []

    for ffile in sorted(fits_files):
        with fits.open(ffile) as hdu:
            h = hdu[0].header

            # Get full local time from header and convert to UTC
            date_obs_local = h['DATE-OBS'].split()[0]  # e.g., '2025-06-01T23:24:17.472'
            t_local = Time(date_obs_local)              # local naive
            t_utc = t_local - 3.5*u.hour               # convert Iran local time to UTC

            # Alt/Az frame
            altaz_frame = AltAz(obstime=t_utc, location=obs_loc)
            altaz = star_coords[star_folder].transform_to(altaz_frame)

            alt_deg = altaz.alt.degree
            zenith_deg = 90 - alt_deg
            airmass = altaz.secz if alt_deg > 0 else None  # only valid above horizon

            # Only keep frames where star is above horizon
            if alt_deg > 0:
                star_results.append({
                    'filename': os.path.basename(ffile),
                    'time_utc': t_utc.iso,
                    'alt_deg': alt_deg,
                    'zenith_deg': zenith_deg,
                    'airmass': airmass
                })

    print(f"Number of valid frames: {len(star_results)}")
    all_results.append({'star': star_folder, 'frames': star_results})

# -----------------------------
# IERS configuration
# -----------------------------
iers.conf.auto_download = False
iers.conf.auto_max_age = None  # allow predictive data for UT1-UTC

# -----------------------------
# Observatory location
# -----------------------------
obs_lat = 33 + 40/60 + 28/3600  # degrees
obs_lon = 51 + 19/60 + 8/3600  # degrees
obs_height = 1  # meters, approximate if unknown
obs_loc = EarthLocation(lat=obs_lat*u.deg, lon=obs_lon*u.deg, height=obs_height*u.m)

# -----------------------------
# Star info (RA/Dec)
# -----------------------------
star_coords = {
    'BD+35_3659': SkyCoord('19h31m09.22s', '+36d09m10.1s', frame='icrs'),
}

# -----------------------------
# Main processing
# -----------------------------
base_folder = r'C:/Users/Observatory/Exo/Data/Standard-stars/BD353659/06_01_2025/BD353659/calibrated'

all_results = []

for star_folder in star_coords.keys():
    fits_files = glob(os.path.join(base_folder, '*.fits'))

    print(f"\nProcessing star {star_folder}: {len(fits_files)} files found")

    star_results = []

    for ffile in sorted(fits_files):
        with fits.open(ffile) as hdu:
            h = hdu[0].header

            # Get full local time from header and convert to UTC
            date_obs_local = h['DATE-OBS'].split()[0]  # e.g., '2025-06-01T23:24:17.472'
            t_local = Time(date_obs_local)              # local naive
            t_utc = t_local - 3.5*u.hour               # convert Iran local time to UTC

            # Alt/Az frame
            altaz_frame = AltAz(obstime=t_utc, location=obs_loc)
            altaz = star_coords[star_folder].transform_to(altaz_frame)

            alt_deg = altaz.alt.degree
            zenith_deg = 90 - alt_deg
            airmass = altaz.secz if alt_deg > 0 else None  # only valid above horizon

            # Only keep frames where star is above horizon
            if alt_deg > 0:
                star_results.append({
                    'filename': os.path.basename(ffile),
                    'time_utc': t_utc.iso,
                    'alt_deg': alt_deg,
                    'zenith_deg': zenith_deg,
                    'airmass': airmass
                })

    print(f"Number of valid frames: {len(star_results)}")
    all_results.append({'star': star_folder, 'frames': star_results})

for res in all_results:
    print(f"\nStar: {res['star']}")
    print(f"{'File':70s} {'Time (UTC)':20s} {'Alt (deg)':>8s} {'Zenith (deg)':>12s} {'Airmass':>8s}")
    for frame in res['frames']:
        fname = os.path.basename(frame['filename'])
        low_index = fname.find('Low')
        fname_short = fname[low_index:] if low_index != -1 else fname

        print(f"{fname_short:70s} {frame['time_utc']:20s} "
              f"{frame['alt_deg']:8.2f} {frame['zenith_deg']:12.2f} {frame['airmass']!s:8s}")


Processing star BD+35_3659: 22 files found
Number of valid frames: 22

Processing star BD+35_3659: 22 files found
Number of valid frames: 22

Star: BD+35_3659
File                                                                   Time (UTC)           Alt (deg) Zenith (deg)  Airmass
Low_1_minus_dark_div_non-dark-reduced_flat_bg2D_calibrated.fits        2025-06-01 19:55:22.086    47.57        42.43 1.3549258096666204
Low_2_minus_dark_div_non-dark-reduced_flat_bg2D_calibrated.fits        2025-06-01 20:03:17.209    49.13        40.87 1.322385227571026
Low_3_minus_dark_div_non-dark-reduced_flat_bg2D_calibrated.fits        2025-06-01 20:12:17.779    50.92        39.08 1.2882521615808542
Low_4_minus_dark_div_non-dark-reduced_flat_bg2D_calibrated.fits        2025-06-01 20:22:33.035    52.96        37.04 1.2528188491374213
Low_1_minus_dark_div_non-dark-reduced_flat_bg2D_calibrated.fits        2025-06-01 20:32:11.420    54.88        35.12 1.222530220275759
Low_2_minus_dark_div_non-dark-reduced_

In [2]:
import csv

output_csv = os.path.join(base_folder, "airmass_results.csv")

with open(output_csv, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["filename", "time_utc", "alt_deg", "zenith_deg", "airmass"])

    for res in all_results:
        for frame in res["frames"]:
            writer.writerow([
                frame["filename"],            # full filename
                frame["time_utc"],
                f"{frame['alt_deg']:.4f}",
                f"{frame['zenith_deg']:.4f}",
                f"{frame['airmass']:.6f}" if frame["airmass"] is not None else ""
            ])

print(f"\nCSV saved to: {output_csv}")



CSV saved to: C:/Users/Observatory/Exo/Data/Standard-stars/BD353659/06_01_2025/BD353659/calibrated\airmass_results.csv
